# YOLO物体検出＋軌跡トラッキング

このノートブックでは、動画をアップロードするだけで物体検出と軌跡トラッキングを実行できます。

## 使い方
1. 上から順番にセルを実行してください（▶ボタンを押す）
2. ファイルアップロード画面で動画ファイルと重みファイルをアップロード
3. 処理完了後、ダウンロードボタンで結果を取得

## ステップ1: 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
print("📦 ライブラリをインストール中...")
!pip install -q ultralytics opencv-python-headless
print("✅ インストール完了！")

# GPU確認
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU利用可能: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU モードで実行します（処理に時間がかかる場合があります）")

## ステップ2: スクリプトファイルを作成

In [ ]:
# tennis.py (前処理スクリプト) を作成
%%writefile tennis.py
import cv2
import sys
import numpy as np

MASK_HEIGHT = 1080
MASK_WIDTH = 1920
MASK_START_X = 0
MASK_START_Y = 0
THRESHOLD = 100
EROS_KERNEL_SIZE = 10
DILATION_SIZE = 5
DETECT_AREA_UPPER = 10000
DETECT_AREA_LOWER = 0


def dilation(dilationSize, kernelSize, img):
    kernel = np.ones((kernelSize, kernelSize), np.uint8)
    element = cv2.getStructuringElement(
        cv2.MORPH_RECT, (5 * dilationSize + 1, 5 * dilationSize + 1), (dilationSize, dilationSize))
    dilation_img = cv2.dilate(img, kernel, element)
    return dilation_img


def detect(gray_diff, thresh_diff=THRESHOLD, dilationSize=DILATION_SIZE, kernelSize=20):
    retval, black_diff = cv2.threshold(
        gray_diff, thresh_diff, 255, cv2.THRESH_BINARY)
    dilation_img = dilation(dilationSize, kernelSize, black_diff)
    img = dilation_img.copy()
    if EROS_KERNEL_SIZE > 0:
        kernel = np.ones((EROS_KERNEL_SIZE, EROS_KERNEL_SIZE), np.uint8)
        erosion = cv2.erode(dilation_img,kernel,iterations = 1)
    else:
        erosion = dilation_img
    contours, hierarchy = cv2.findContours(
        erosion, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    ball_pos = []

    for i in range(len(contours)):
        count = len(contours[i])
        area = cv2.contourArea(contours[i])
        x, y = 0.0, 0.0
        for j in range(count):
            x += contours[i][j][0][0]
            y += contours[i][j][0][1]

        x /= count
        y /= count
        x = int(x)
        y = int(y)
        if int(area) > DETECT_AREA_UPPER or int(area) < DETECT_AREA_LOWER :
            break
        ball_pos.append([x, y, area])

    return ball_pos, img


def displayCircle(image, ballList, thickness):
    overlay = image.copy()
    for i in range(len(ballList)):
        x = int(ballList[i][0])
        y = int(ballList[i][1])
        area = int(ballList[i][2])
        cv2.circle(image, (x, y), 10, (0, 0, 255), thickness)
        image = cv2.addWeighted(overlay, 0.3, image, 0.7, 0)
    return image


def resizeImage(image, w=2, h=2):
    height = image.shape[0]
    width = image.shape[1]
    resizedImage = cv2.resize(image, (int(width / w), int(height / h)))
    return resizedImage


def blackToColor(bImage):
    colorImage = np.array((bImage, bImage, bImage))
    colorImage = colorImage.transpose(1, 2, 0)
    return colorImage


def run(input_video_path, output_video_path=None, masked_video_path=None, enhance_video_path=None):
    """
    動画の前処理を実行

    Args:
        input_video_path: 入力動画のパス
        output_video_path: 検出円を描画した動画の出力パス（Noneの場合は生成しない）
        masked_video_path: 差分動画の出力パス（Noneの場合は生成しない）
        enhance_video_path: 強調動画の出力パス（Noneの場合は生成しない）

    Returns:
        生成されたファイルのパスの辞書
    """
    video = cv2.VideoCapture(input_video_path)
    outFourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = video.get(cv2.CAP_PROP_FPS)

    if not video.isOpened():
        print("Could not open video")
        sys.exit()

    vidw = video.get(cv2.CAP_PROP_FRAME_WIDTH)
    vidh = video.get(cv2.CAP_PROP_FRAME_HEIGHT)
    print(vidw, vidh)

    out = cv2.VideoWriter(output_video_path, outFourcc, fps,
                          (int(vidw), int(vidh))) if output_video_path else None
    out2 = cv2.VideoWriter(masked_video_path, outFourcc, fps,
                          (int(vidw), int(vidh))) if masked_video_path else None
    out3 = cv2.VideoWriter(enhance_video_path, outFourcc, fps,
                          (int(vidw), int(vidh))) if enhance_video_path else None

    ok, frame = video.read()
    if not ok:
        print('Cannot read video file')
        sys.exit()

    frame_pre = frame.copy()
    frame_count = 1
    while True:
        frame_count += 1
        frame_next = frame.copy()
        color_diff = cv2.absdiff(frame_next, frame_pre)
        ok, frame4 = video.read()
        if not ok:
            break
        color_diff2 = cv2.absdiff(frame4, frame_next)
        im_mask = np.zeros((int(vidh), int(vidw), 3), np.uint8)
        im_mask = cv2.rectangle(im_mask, (MASK_START_X, MASK_START_Y), (
            MASK_START_X + MASK_WIDTH, MASK_START_Y + MASK_HEIGHT), (255, 255, 255), -1)
        im_mask = cv2.cvtColor(im_mask, cv2.COLOR_BGR2GRAY)
        diff = cv2.bitwise_and(color_diff, color_diff2)
        gamma = 1.5
        look_up_table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in range(256)]).astype("uint8")
        diff = cv2.LUT(diff, look_up_table)
        enhanced = cv2.addWeighted(frame, 0.4, diff, 0.6, 0)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        gray_diff_m = cv2.bitwise_and(gray_diff, im_mask)
        retval, black_diff = cv2.threshold(
            gray_diff_m, 30, 255, cv2.THRESH_BINARY)

        ball, dilation_img = detect(gray_diff_m)

        frame = displayCircle(frame, ball, -1)
        cImage = blackToColor(dilation_img)
        frame_pre = frame_next
        print(frame_count)
        if out:
            out.write(frame)
        if out2:
            out2.write(diff)
        if out3:
            out3.write(enhanced)

        frame = frame4
    video.release()
    if out:
        out.release()
    if out2:
        out2.release()
    if out3:
        out3.release()

    print(f"\nProcessing completed!")
    result_paths = {}
    if output_video_path:
        print(f"Output saved to: {output_video_path}")
        result_paths['output'] = output_video_path
    if masked_video_path:
        print(f"Masked saved to: {masked_video_path}")
        result_paths['masked'] = masked_video_path
    if enhance_video_path:
        print(f"Enhanced saved to: {enhance_video_path}")
        result_paths['enhanced'] = enhance_video_path

    return result_paths

In [ ]:
# track.py (検出＋トラッキングスクリプト) を作成
%%writefile track.py
import cv2
import sys
import os
import numpy as np
from ultralytics import YOLO
import tennis

def run_tracking(original_video, model_path, target_classes=[0]):
    # ファイル名とディレクトリを取得
    base_name = os.path.splitext(os.path.basename(original_video))[0]
    video_dir = os.path.dirname(original_video) if os.path.dirname(original_video) else "."

    # 強調動画のパスを設定
    enhance_video = os.path.join(video_dir, f"{base_name}_enhance.mp4")

    print(f"Original video: {original_video}")
    print(f"Generating enhanced video for detection...")

    # tennisモジュールで強調動画を生成
    tennis.run(original_video, enhance_video_path=enhance_video)

    print(f"\nLoading YOLO model...")
    # YOLOモデルをロード
    model = YOLO(model_path)

    # 軌跡描画設定
    DRAW_TRAJECTORY = True
    MAX_TRAJECTORY_LENGTH = 10
    TRAJECTORY_FADE_FRAMES = 15
    TRAJECTORY_COLOR = (0, 255, 0)
    TRAJECTORY_THICKNESS = 4

    trajectories = {}
    next_object_id = 0

    # 強調動画と元動画を開く
    cap_enhance = cv2.VideoCapture(enhance_video)
    cap_original = cv2.VideoCapture(original_video)

    # 動画情報を取得
    fps = cap_original.get(cv2.CAP_PROP_FPS)
    width = int(cap_original.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap_original.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 出力動画の設定
    output_path = os.path.join(video_dir, f"{base_name}_detected.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"\nProcessing frames and detecting objects...")

    frame_count = 0
    while True:
        ret_enhance, frame_enhance = cap_enhance.read()
        ret_original, frame_original = cap_original.read()

        if not ret_enhance or not ret_original:
            break

        # 強調フレームで検出実行
        results = model(frame_enhance, verbose=False)

        # 現在フレームの検出物体の中心座標を取得
        current_centers = []

        # 検出結果を元フレームに描画
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if target_classes and cls not in target_classes:
                    continue

                center_x = int((x1 + x2) / 2)
                center_y = int((y1 + y2) / 2)
                current_centers.append((center_x, center_y, cls))

                cv2.rectangle(frame_original, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"{model.names[cls]} {conf:.2f}"
                cv2.putText(frame_original, label, (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # 軌跡を更新
        if DRAW_TRAJECTORY:
            used_centers = set()
            for obj_id in list(trajectories.keys()):
                if not current_centers:
                    break

                points = trajectories[obj_id]['points']
                last_point = points[-1] if points else None
                if last_point is None:
                    continue

                min_dist = float('inf')
                best_match = None
                for i, (cx, cy, cls) in enumerate(current_centers):
                    if i in used_centers:
                        continue
                    dist = np.sqrt((last_point[0] - cx)**2 + (last_point[1] - cy)**2)
                    if dist < min_dist and dist < 100:
                        min_dist = dist
                        best_match = i

                if best_match is not None:
                    cx, cy, cls = current_centers[best_match]
                    trajectories[obj_id]['points'].append((cx, cy))
                    trajectories[obj_id]['last_seen'] = frame_count
                    used_centers.add(best_match)
                    if len(trajectories[obj_id]['points']) > MAX_TRAJECTORY_LENGTH:
                        trajectories[obj_id]['points'].pop(0)

            for i, (cx, cy, cls) in enumerate(current_centers):
                if i not in used_centers:
                    trajectories[next_object_id] = {
                        'points': [(cx, cy)],
                        'last_seen': frame_count
                    }
                    next_object_id += 1

            trajectories_to_remove = []
            for obj_id, traj_data in trajectories.items():
                if frame_count - traj_data['last_seen'] > TRAJECTORY_FADE_FRAMES:
                    trajectories_to_remove.append(obj_id)
            for obj_id in trajectories_to_remove:
                del trajectories[obj_id]

            for obj_id, traj_data in trajectories.items():
                points = traj_data['points']
                frames_since_seen = frame_count - traj_data['last_seen']
                fade_alpha = max(0, 1 - (frames_since_seen / TRAJECTORY_FADE_FRAMES))

                if len(points) > 1 and fade_alpha > 0:
                    for i in range(1, len(points)):
                        alpha = (i / len(points)) * fade_alpha
                        thickness = max(1, int(TRAJECTORY_THICKNESS * alpha))
                        color = tuple(int(c * fade_alpha) for c in TRAJECTORY_COLOR)
                        cv2.line(frame_original, points[i-1], points[i], color, thickness)

        out.write(frame_original)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Processed {frame_count} frames")

    cap_enhance.release()
    cap_original.release()
    out.release()

    print(f"\nDetection completed!")
    print(f"Output saved to: {output_path}")
    return output_path

## ステップ3: ファイルをアップロード

**必要なファイル:**
1. **動画ファイル** (.mp4など)
2. **重みファイル** (yolo8m_20250510.pt など)

下のセルを実行すると、ファイル選択画面が表示されます。

In [ ]:
from google.colab import files
import os

print("📹 動画ファイルをアップロードしてください")
video_uploaded = files.upload()
video_filename = list(video_uploaded.keys())[0]
print(f"✅ 動画アップロード完了: {video_filename}")

print("\n⚖️ 重みファイル（.ptファイル）をアップロードしてください")
model_uploaded = files.upload()
model_filename = list(model_uploaded.keys())[0]
print(f"✅ 重みファイルアップロード完了: {model_filename}")

## ステップ4: 検出対象クラスの設定（必要に応じて変更）

検出したい物体のクラスIDを指定します。

**よく使うクラスID:**
- `[0]` - 人（person）
- `[32]` - スポーツボール（sports ball）
- `[37]` - テニスラケット（tennis racket）
- `[]` - 全クラス検出
- `[0, 32, 37]` - 複数指定

In [ ]:
# 検出対象クラスを指定（変更可能）
target_classes = [0]  # 人のみ検出

print(f"検出対象クラス: {target_classes}")

## ステップ5: 実行

このセルを実行すると、動画の処理が開始されます。
処理時間は動画の長さによって変わります（数分〜数十分）。

In [ ]:
import track

print("🚀 処理を開始します...\n")

# トラッキング実行
output_video = track.run_tracking(
    original_video=video_filename,
    model_path=model_filename,
    target_classes=target_classes
)

print("\n🎉 処理完了！")

## ステップ6: 結果をダウンロード

処理が完了したら、このセルを実行して結果動画をダウンロードします。

In [ ]:
from google.colab import files

print("📥 結果動画をダウンロード中...")
files.download(output_video)
print("✅ ダウンロード完了！")

## オプション: 中間ファイルもダウンロード

強調動画も確認したい場合は、このセルを実行してください。

In [ ]:
import os
from google.colab import files

base_name = os.path.splitext(video_filename)[0]
enhance_video = f"{base_name}_enhance.mp4"

if os.path.exists(enhance_video):
    print(f"📥 強調動画をダウンロード中: {enhance_video}")
    files.download(enhance_video)
    print("✅ ダウンロード完了！")
else:
    print("強調動画が見つかりません")